# AION-U1 — 1-Epochen-Lauf (Colab T4)

Trainiert den Adapter mit `training/configs/sft_lora_open_e1.yaml`
(1 Epoche, 63 Saetze, fp32) und misst ihn gegen die Baseline.
**Laufzeittyp T4/GPU waehlen** — auf CPU steht der Lauf bei Schritt 0.

Ziel: pruefen, ob 1 Epoche die Endlosschleifen-Hypothese bestaetigt
(Baseline 8/24, v3/3-Epochen 12/24 mit 3 length-Abbruechen).

In [ ]:
# 1) Repo
import os, subprocess
if not os.path.isdir('aion-u1'):
    subprocess.run(['git','clone','-q','-b','feat/pilot-ergebnis-v2','https://github.com/Reznap87/aion-u1.git'], check=True)
%cd aion-u1
!pip install -q -e '.[train]'
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Bitte Laufzeittyp T4/GPU waehlen — CPU ist zu langsam'

In [ ]:
# 2) Trainingsconfig pruefen
from pathlib import Path
cfg = Path('training/configs/sft_lora_open_e1.yaml').read_text(encoding='utf-8')
print('Epochen 1:', 'num_train_epochs: 1' in cfg, '| fp32:', 'fp16: false' in cfg)
print('Daten:', Path('data/verified/pilot_open_train.jsonl').exists())

In [ ]:
# 3) Training (1 Epoche, Ausgabe artifacts/sft/pilot_open_e1)
import pathlib, subprocess
ziel = pathlib.Path('artifacts/sft/pilot_open_e1/final_model')
if ziel.is_dir():
    print('Adapter vorhanden:', ziel)
else:
    subprocess.run(['aion-u1-train-sft','--config','training/configs/sft_lora_open_e1.yaml'], check=True)
print('final_model existiert:', ziel.is_dir())

In [ ]:
# 4) Baseline (untrainiert, gleiche Suite)
import subprocess
subprocess.run(['aion-u1-evaluate-model', '--base-model', 'Qwen/Qwen2.5-1.5B-Instruct',
                 '--suite', 'evaluations/capability_suite/benchmark_hard_text_v1.yaml',
                 '--device', 'auto', '--max-tokens', '512',
                 '--output', 'reports/eval-baseline.json'], check=True)

In [ ]:
# 5) Messung des 1-Epochen-Adapters
import subprocess
subprocess.run(['aion-u1-evaluate-model', '--base-model', 'Qwen/Qwen2.5-1.5B-Instruct',
                 '--adapter', 'artifacts/sft/pilot_open_e1/final_model',
                 '--suite', 'evaluations/capability_suite/benchmark_hard_text_v1.yaml',
                 '--device', 'auto', '--max-tokens', '512',
                 '--output', 'reports/eval-pilot-e1.json'], check=True)

In [ ]:
# 6) Ergebnis + Vergleich mit den bekannten Laeufen
import json
for name in ('eval-baseline', 'eval-pilot-e1'):
    try:
        d = json.load(open(f'reports/{name}.json', encoding='utf-8'))
        s = list(d['scores'].values())[0]
        r = d['results']
        print(f"{name:16} korrekt={sum(1 for x in r if x.get('score')==1.0):2}/24 "
              f"coverage={s.get('coverage')} index={s.get('overall_score')} "
              f"length={sum(1 for x in r if x.get('finish_reason')=='length')}")
    except FileNotFoundError:
        print(name, 'fehlt')
print('bekannt: baseline 8/24, v3/3-Epochen 12/24 (3 length)')

In [ ]:
# 7) Adapter sichern
import os
if os.path.isdir('artifacts/sft/pilot_open_e1/final_model'):
    !zip -q -r sft_pilot_e1.zip artifacts/sft/pilot_open_e1/final_model
    from google.colab import files
    files.download('sft_pilot_e1.zip')
else:
    print('kein Adapter vorhanden')